# Assignment 03 — Bài toán 2: Định giá Bất động sản bằng Mạng nơ-ron sâu 5 tầng

**Môn học:** Intelligent System Development — TS. Trần Đình Quế
**Sinh viên:** Đinh Hải Triều — B23DCCN843 — Lớp 06

---

## Mục tiêu

1. Cài đặt **mạng nơ-ron sâu 5 tầng** thuần **NumPy** cho bài toán **hồi quy**
   (tầng 5 dùng kích hoạt tuyến tính + hàm mất mát MSE) và bản **PyTorch** đối chiếu.
2. Chỉ ra vì sao phải **học trên thang log** (`log1p(Price)`) thay vì giá gốc,
   và điều đó ảnh hưởng thế nào tới MAE/RMSE khi quy đổi ngược.
3. Đối sánh với Linear Regression, Decision Tree, Gradient Boosting.
4. Xuất trọng số ra `model_deep.json` và nhúng vào web app chạy trên Vercel.

**Bài toán:** Regression — dự đoán `Price` (tỉ VNĐ) của bất động sản Việt Nam.

In [1]:
# ============================================================================
# KHỐI 1 — Nạp thư viện, cố định seed và cấu hình hình vẽ
# Giống Bài toán 1 nhưng import các độ đo hồi quy (MAE, MSE, R2) thay cho
# các độ đo phân loại. Hình vẽ lưu vào figures/.
# ============================================================================
import json
import time
import pathlib

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 150, "font.family": "DejaVu Sans",
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
})

ROOT = pathlib.Path.cwd()
FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)
print("Thư mục làm việc:", ROOT)

Thư mục làm việc: C:\Users\admin\Downloads\bt-thay quế\tuan 1\website_dudoan_gia_nha


## 1. Nạp dữ liệu và khảo sát

Bộ dữ liệu bất động sản Việt Nam: 2000 bản ghi, 6 đặc trưng (2 phân loại +
4 số), nhãn `Price` tính bằng **tỉ VNĐ**.

In [2]:
# ============================================================================
# KHỐI 2 — Nạp bộ dữ liệu bất động sản Việt Nam
# 2000 bản ghi, 6 đặc trưng (2 phân loại + 4 số), nhãn Price tính bằng tỉ VNĐ.
# Kiểm tra nhanh kiểu dữ liệu và số ô trống trước khi xử lý.
# ============================================================================
df = pd.read_csv(ROOT / "data" / "vietnam_housing_dataset.csv")
print("Kích thước:", df.shape)
print("\nKiểu dữ liệu:")
print(df.dtypes.to_string())
print("\nGiá trị thiếu:", int(df.isna().sum().sum()))
df.head()

Kích thước: (2000, 7)

Kiểu dữ liệu:
Province         str
Area         float64
Bedrooms       int64
Bathrooms      int64
Floors         int64
HouseType        str
Price        float64

Giá trị thiếu: 0


,Province,Area,Bedrooms,Bathrooms,Floors,HouseType,Price
0,Đồng Nai,29.3,1,1,3,Nhà phố,6.113
1,Hải Phòng,27.5,3,3,2,Đất nền,4.713
2,Khánh Hòa,168.3,4,4,1,Chung cư,50.060
3,Cần Thơ,65.9,4,5,1,Nhà phố,15.835
4,Đồng Nai,33.5,2,3,2,Nhà phố,8.961


In [3]:
# ============================================================================
# KHỐI 3 — Khảo sát độ lệch (skewness) của biến mục tiêu
# So sánh hệ số bất đối xứng của Price và của log1p(Price):
# giá gốc lệch phải rất mạnh, sau khi lấy log thì gần như đối xứng.
# Đây là căn cứ để huấn luyện mô hình trên thang log thay vì thang giá gốc.
# ============================================================================
print("Thống kê giá (tỉ VNĐ):")
print(df["Price"].describe().round(3).to_string())
print(f"\nHệ số bất đối xứng (skewness) của Price       : {df['Price'].skew():.3f}")
print(f"Hệ số bất đối xứng của log1p(Price)           : {np.log1p(df['Price']).skew():.3f}")
print(f"Tỉ lệ giá lớn nhất / giá trung vị              : {df['Price'].max()/df['Price'].median():.1f} lần")
df.describe().T[["mean", "std", "min", "50%", "max"]].round(2)

Thống kê giá (tỉ VNĐ):
count    2000.000
mean       20.126
std        17.675
min         1.617
25%         8.818
50%        14.764
75%        25.421
max       200.000

Hệ số bất đối xứng (skewness) của Price       : 2.949
Hệ số bất đối xứng của log1p(Price)           : 0.279
Tỉ lệ giá lớn nhất / giá trung vị              : 13.5 lần


,mean,std,min,50%,max
Area,109.60,71.05,20.00,92.15,500.0
Bedrooms,2.78,1.06,1.00,3.00,5.0
Bathrooms,2.79,1.22,1.00,3.00,5.0
Floors,2.25,1.10,1.00,2.00,5.0
Price,20.13,17.67,1.62,14.76,200.0


In [4]:
# ============================================================================
# KHỐI 4 — Hình 1 — bốn biểu đồ khảo sát dữ liệu
# (a) phân bố giá gốc kèm đường trung vị và trung bình (lệch nhau nhiều = lệch phải).
# (b) phân bố log1p(Price) gần chuẩn.
# (c) boxplot giá theo loại hình trên thang log.
# (d) scatter log–log diện tích vs giá, tô màu theo số phòng ngủ.
# ============================================================================
fig, axes = plt.subplots(2, 2, figsize=(13.5, 9))

# (a) Phân bố giá gốc
axes[0, 0].hist(df["Price"], bins=60, color="#10b981", edgecolor="white")
axes[0, 0].axvline(df["Price"].median(), color="#111827", ls="--",
                   label=f"Trung vị = {df['Price'].median():.1f}")
axes[0, 0].axvline(df["Price"].mean(), color="#ef4444", ls="--",
                   label=f"Trung bình = {df['Price'].mean():.1f}")
axes[0, 0].set_title(f"(a) Phân bố Price — lệch phải mạnh (skew={df['Price'].skew():.2f})",
                     fontweight="bold")
axes[0, 0].set_xlabel("Giá (tỉ VNĐ)"); axes[0, 0].set_ylabel("Số bất động sản"); axes[0, 0].legend()

# (b) Phân bố log
axes[0, 1].hist(np.log1p(df["Price"]), bins=60, color="#6366f1", edgecolor="white")
axes[0, 1].set_title(f"(b) Phân bố log1p(Price) — gần chuẩn (skew={np.log1p(df['Price']).skew():.2f})",
                     fontweight="bold")
axes[0, 1].set_xlabel("log1p(Giá)"); axes[0, 1].set_ylabel("Số bất động sản")

# (c) Giá theo loại hình
order = df.groupby("HouseType")["Price"].median().sort_values().index
sns.boxplot(data=df, x="HouseType", y="Price", order=order, ax=axes[1, 0],
            palette="viridis", showfliers=False, hue="HouseType", legend=False)
axes[1, 0].set_yscale("log")
axes[1, 0].set_title("(c) Giá theo loại hình (thang log)", fontweight="bold")
axes[1, 0].set_xlabel(""); axes[1, 0].set_ylabel("Giá (tỉ VNĐ, log)")
axes[1, 0].tick_params(axis="x", rotation=20)

# (d) Diện tích vs giá
sc = axes[1, 1].scatter(df["Area"], df["Price"], c=df["Bedrooms"], cmap="plasma",
                        s=14, alpha=0.6)
axes[1, 1].set_xscale("log"); axes[1, 1].set_yscale("log")
axes[1, 1].set_title("(d) Diện tích vs Giá (log–log), màu = số phòng ngủ", fontweight="bold")
axes[1, 1].set_xlabel("Diện tích (m², log)"); axes[1, 1].set_ylabel("Giá (tỉ VNĐ, log)")
plt.colorbar(sc, ax=axes[1, 1], label="Bedrooms")

plt.tight_layout()
plt.savefig(FIG / "p2_fig1_eda.png", bbox_inches="tight")
plt.show()

C:\Users\admin\AppData\Local\Temp\ipykernel_3052\992651604.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Tiền xử lý

### 2.1. One-Hot Encoding thay vì Label Encoding

`Province` và `HouseType` là biến **định danh** (nominal) — gán số 0..9 sẽ áp
một thứ tự giả lên mạng ("Hà Nội = 3 < TP.HCM = 7"). Với mạng nơ-ron ta dùng
**one-hot**: mỗi hạng mục thành một chiều riêng, tầng 1 tự học một vector nhúng
cho từng tỉnh.

### 2.2. Học trên thang log

Giá nhà lệch phải rất mạnh. Nếu tối thiểu hoá MSE trên giá gốc, một biệt thự
300 tỉ đóng góp sai số bình phương gấp hàng nghìn lần một căn trọ 2 tỉ — mạng
sẽ chỉ tối ưu cho nhóm siêu đắt. Học `log1p(Price)` biến sai số tuyệt đối
thành **sai số tương đối**, đúng với cách thị trường định giá.

In [5]:
# ============================================================================
# KHỐI 5 — One-Hot Encoding và dựng ma trận đặc trưng
# Province và HouseType là biến ĐỊNH DANH (nominal) — không có thứ tự lớn/nhỏ,
# nên dùng one-hot (np.eye) thay vì gán số 0,1,2... để mô hình không hiểu nhầm
# rằng Hà Nội nhỏ hơn TP.HCM.
# Kết quả: 4 cột số + 10 cột tỉnh + 5 cột loại hình = 19 chiều đầu vào.
# ============================================================================
CAT = ["Province", "HouseType"]
NUM = ["Area", "Bedrooms", "Bathrooms", "Floors"]

provinces = sorted(df["Province"].unique())
house_types = sorted(df["HouseType"].unique())
print(f"Province ({len(provinces)}): {provinces}")
print(f"HouseType ({len(house_types)}): {house_types}")

X_num = df[NUM].values.astype(float)
X_cat = np.hstack([
    np.eye(len(provinces))[[provinces.index(v) for v in df["Province"]]],
    np.eye(len(house_types))[[house_types.index(v) for v in df["HouseType"]]],
])
X_all = np.hstack([X_num, X_cat])
FEATURE_NAMES = NUM + [f"Province={p}" for p in provinces] + [f"HouseType={t}" for t in house_types]
y_all = df["Price"].values.astype(float).reshape(-1, 1)

print(f"\nMa trận đặc trưng sau one-hot: {X_all.shape}  "
      f"({len(NUM)} số + {len(provinces)} tỉnh + {len(house_types)} loại hình)")

Province (10): ['Bà Rịa-VT', 'Bình Dương', 'Cần Thơ', 'Hà Nội', 'Hải Phòng', 'Khánh Hòa', 'Quảng Ninh', 'TP.HCM', 'Đà Nẵng', 'Đồng Nai']
HouseType (5): ['Biệt thự', 'Chung cư', 'Nhà phố', 'Nhà trọ', 'Đất nền']

Ma trận đặc trưng sau one-hot: (2000, 19)  (4 số + 10 tỉnh + 5 loại hình)


In [6]:
# ============================================================================
# KHỐI 6 — Chia dữ liệu, chuẩn hoá đầu vào và biến đổi nhãn
# Chia 70/15/15. Scaler chỉ fit trên train và CHỈ áp dụng cho 4 cột số đầu tiên —
# các cột one-hot đã là 0/1 nên giữ nguyên.
# Nhãn được biến đổi hai bước: log1p để bớt lệch, rồi z-score (trừ mean chia std
# tính trên train) để mạng học ổn định với learning rate thông thường.
# Hàm to_price() làm đúng phép biến đổi ngược: z -> log1p -> giá thật (tỉ VNĐ).
# ============================================================================
X_tmp, X_test, y_tmp, y_test = train_test_split(X_all, y_all, test_size=0.15, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.1765, random_state=SEED)

# Chuẩn hoá: chỉ 4 cột số đầu tiên (one-hot giữ nguyên 0/1)
scaler = StandardScaler().fit(X_train[:, :len(NUM)])
def prep(X):
    out = X.copy()
    out[:, :len(NUM)] = scaler.transform(X[:, :len(NUM)])
    return out

Xtr, Xva, Xte = prep(X_train), prep(X_val), prep(X_test)

# Nhãn: log1p rồi chuẩn hoá về trung bình 0 / độ lệch chuẩn 1
ytr_log = np.log1p(y_train)
Y_MEAN, Y_STD = float(ytr_log.mean()), float(ytr_log.std())
ztr = (ytr_log - Y_MEAN) / Y_STD
zva = (np.log1p(y_val) - Y_MEAN) / Y_STD
zte = (np.log1p(y_test) - Y_MEAN) / Y_STD

print(f"Train {Xtr.shape} | Val {Xva.shape} | Test {Xte.shape}")
print(f"log1p(Price) trên train: mean={Y_MEAN:.4f}, std={Y_STD:.4f}")

def to_price(z):
    """Quy đổi ngược: z-score -> log1p -> giá thật (tỉ VNĐ)."""
    return np.expm1(z * Y_STD + Y_MEAN)

Train (1399, 19) | Val (301, 19) | Test (300, 19)
log1p(Price) trên train: mean=2.7900, std=0.6870


## 3. Mạng nơ-ron sâu 5 tầng (NumPy from scratch)

```
Input(19) → [W1] 128 → ReLU → [W2] 64 → ReLU → [W3] 32 → ReLU
          → [W4] 16 → ReLU → [W5] 1 → Linear   (loss = MSE)
```

Khác biệt duy nhất so với Bài toán 1 nằm ở **tầng 5**: bỏ Sigmoid, đổi
Binary Cross-Entropy thành MSE. Gradient tại tầng cuối vẫn giữ dạng
$\partial\mathcal{L}/\partial Z^{[5]} = 2(\hat y - y)/n$.

In [7]:
# ============================================================================
# KHỐI 7 — Thư viện mạng nơ-ron sâu 5 tầng viết thuần NumPy (mlp_numpy.py)
# Toàn bộ phần lõi của bài: không dùng framework, tự cài đặt từ đầu gồm
#   (1) hàm kích hoạt ReLU/Sigmoid/Softmax và đạo hàm tương ứng,
#   (2) lớp DeepMLP: forward, hàm mất mát, backward (lan truyền ngược),
#       bộ tối ưu Adam, vòng lặp fit theo mini-batch và early stopping,
#   (3) tiện ích one_hot và forward_reference (bản suy luận tham chiếu
#       dùng để đối chiếu kết quả giữa NumPy, JSON và JavaScript).
# Chi tiết từng bước được chú thích trực tiếp trong thân hàm bên dưới.
# ============================================================================
from __future__ import annotations

import numpy as np

# ----------------------------------------------------------------------------
# 1. Hàm kích hoạt và đạo hàm
# ----------------------------------------------------------------------------


def relu(z):
    """f(z) = max(0, z) — phá vỡ tính tuyến tính, giữ gradient không bão hoà ở nhánh dương."""
    return np.maximum(0.0, z)


def relu_grad(z):
    """f'(z) = 1 nếu z > 0, ngược lại 0."""
    return (z > 0).astype(z.dtype)


def sigmoid(z):
    """Ổn định số học: tách nhánh z >= 0 và z < 0 để tránh exp() tràn số."""
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out


def softmax(z):
    """Trừ max theo hàng trước khi exp — kỹ thuật log-sum-exp chống tràn số."""
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


# ----------------------------------------------------------------------------
# 2. Mạng nơ-ron sâu 5 tầng
# ----------------------------------------------------------------------------


class DeepMLP:
    """5-Layer MLP thuần NumPy với Adam, mini-batch, He-init, L2 và Dropout."""

    def __init__(
        self,
        input_dim: int,
        hidden=(128, 64, 32, 16),
        output_dim: int = 1,
        task: str = "binary",
        lr: float = 1e-3,
        l2: float = 1e-4,
        dropout: float = 0.0,
        seed: int = 42,
        class_weight=None,
    ):
        assert task in {"binary", "regression", "multiclass"}
        self.task = task
        # class_weight: vector trọng số theo lớp, dùng để chống mất cân bằng dữ liệu.
        # Mỗi mẫu được nhân thêm w[y] trong cả hàm mất mát lẫn gradient, tương đương
        # "nhân bản" mẫu của lớp hiếm mà không phải sao chép dữ liệu thật.
        self.class_weight = None if class_weight is None else np.asarray(class_weight, dtype=float)
        self.lr = lr
        self.l2 = l2
        self.dropout = dropout
        self.dims = [input_dim, *hidden, output_dim]
        self.rng = np.random.default_rng(seed)

        # --- Khởi tạo He (Kaiming): Var(W) = 2/fan_in, phù hợp với ReLU ---
        self.W, self.b = [], []
        for i in range(len(self.dims) - 1):
            fan_in, fan_out = self.dims[i], self.dims[i + 1]
            self.W.append(self.rng.normal(0.0, np.sqrt(2.0 / fan_in), (fan_in, fan_out)))
            self.b.append(np.zeros(fan_out))

        # --- Trạng thái Adam (moment bậc 1 và bậc 2) ---
        self.mW = [np.zeros_like(w) for w in self.W]
        self.vW = [np.zeros_like(w) for w in self.W]
        self.mb = [np.zeros_like(b) for b in self.b]
        self.vb = [np.zeros_like(b) for b in self.b]
        self.t = 0

        self.history = {"train_loss": [], "val_loss": [], "train_metric": [], "val_metric": []}

    # ----------------------------- forward ---------------------------------
    def forward(self, X, training: bool = False):
        """Trả về (output, cache). cache giữ A/Z của từng tầng để lan truyền ngược."""
        A = X
        As, Zs, masks = [A], [], []
        n_layers = len(self.W)

        for i in range(n_layers - 1):  # 4 tầng ẩn
            Z = A @ self.W[i] + self.b[i]
            A = relu(Z)
            if training and self.dropout > 0.0:
                # Inverted dropout: chia cho keep_prob ngay lúc train nên lúc
                # suy luận không cần chỉnh gì — trọng số xuất ra JSON dùng trực tiếp.
                keep = 1.0 - self.dropout
                mask = (self.rng.random(A.shape) < keep) / keep
                A = A * mask
                masks.append(mask)
            else:
                masks.append(None)
            Zs.append(Z)
            As.append(A)

        # Tầng 5 — output head
        Zout = A @ self.W[-1] + self.b[-1]
        Zs.append(Zout)
        if self.task == "binary":
            out = sigmoid(Zout)
        elif self.task == "multiclass":
            out = softmax(Zout)
        else:
            out = Zout
        As.append(out)
        return out, (As, Zs, masks)

    # ------------------------------ loss -----------------------------------
    def _sample_w(self, y_true):
        """Trọng số của từng mẫu suy ra từ class_weight (shape (n, 1))."""
        if self.class_weight is None or self.task != "multiclass":
            return None
        return self.class_weight[y_true.argmax(1)].reshape(-1, 1)

    def loss(self, y_pred, y_true):
        n = y_true.shape[0]
        if self.task == "binary":
            eps = 1e-12
            base = -np.mean(
                y_true * np.log(y_pred + eps) + (1 - y_true) * np.log(1 - y_pred + eps)
            )
        elif self.task == "multiclass":
            eps = 1e-12
            per_sample = -np.sum(y_true * np.log(y_pred + eps), axis=1, keepdims=True)
            w = self._sample_w(y_true)
            base = float(np.sum(per_sample if w is None else per_sample * w) / n)
        else:
            base = np.mean((y_pred - y_true) ** 2)
        reg = self.l2 * sum(np.sum(w * w) for w in self.W) / (2 * n)
        return base + reg

    # ---------------------------- backward ---------------------------------
    def backward(self, cache, y_true):
        As, Zs, masks = cache
        n = y_true.shape[0]
        n_layers = len(self.W)

        # Với cả 3 head, đạo hàm của loss theo pre-activation cuối cùng rút gọn
        # về (y_hat - y). Đây là lý do ta ghép Sigmoid/Softmax với Cross-Entropy
        # và Linear với MSE.
        dZ = (As[-1] - y_true) / n
        if self.task == "regression":
            dZ = 2.0 * dZ
        w = self._sample_w(y_true)
        if w is not None:
            dZ = dZ * w

        dW = [None] * n_layers
        db = [None] * n_layers

        for i in range(n_layers - 1, -1, -1):
            dW[i] = As[i].T @ dZ + self.l2 * self.W[i] / n
            db[i] = dZ.sum(axis=0)
            if i > 0:
                dA = dZ @ self.W[i].T
                if masks[i - 1] is not None:
                    dA = dA * masks[i - 1]
                dZ = dA * relu_grad(Zs[i - 1])
        return dW, db

    # ------------------------------ Adam -----------------------------------
    def _adam(self, dW, db, beta1=0.9, beta2=0.999, eps=1e-8):
        self.t += 1
        for i in range(len(self.W)):
            self.mW[i] = beta1 * self.mW[i] + (1 - beta1) * dW[i]
            self.vW[i] = beta2 * self.vW[i] + (1 - beta2) * (dW[i] ** 2)
            mhat = self.mW[i] / (1 - beta1**self.t)
            vhat = self.vW[i] / (1 - beta2**self.t)
            self.W[i] -= self.lr * mhat / (np.sqrt(vhat) + eps)

            self.mb[i] = beta1 * self.mb[i] + (1 - beta1) * db[i]
            self.vb[i] = beta2 * self.vb[i] + (1 - beta2) * (db[i] ** 2)
            mhat = self.mb[i] / (1 - beta1**self.t)
            vhat = self.vb[i] / (1 - beta2**self.t)
            self.b[i] -= self.lr * mhat / (np.sqrt(vhat) + eps)

    # ------------------------------ metric ---------------------------------
    def _metric(self, X, y):
        p, _ = self.forward(X, training=False)
        if self.task == "binary":
            return float(np.mean((p >= 0.5).astype(int) == y))
        if self.task == "multiclass":
            return float(np.mean(p.argmax(1) == y.argmax(1)))
        ss_res = np.sum((y - p) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        return float(1 - ss_res / ss_tot)  # R^2

    # ------------------------------- fit -----------------------------------
    def fit(self, X, y, X_val=None, y_val=None, epochs=200, batch_size=32,
            verbose_every=20, patience=None):
        """Chu trình huấn luyện 4 bước: Forward -> Loss -> Backward -> Update."""
        n = X.shape[0]
        best_val, best_state, wait = np.inf, None, 0

        for ep in range(1, epochs + 1):
            idx = self.rng.permutation(n)
            Xs, ys = X[idx], y[idx]

            for s in range(0, n, batch_size):
                xb, yb = Xs[s:s + batch_size], ys[s:s + batch_size]
                _, cache = self.forward(xb, training=True)          # (1) Forward
                dW, db = self.backward(cache, yb)                   # (3) Backward
                self._adam(dW, db)                                  # (4) Update

            tr_pred, _ = self.forward(X, training=False)
            tr_loss = self.loss(tr_pred, y)                         # (2) Loss
            self.history["train_loss"].append(tr_loss)
            self.history["train_metric"].append(self._metric(X, y))

            if X_val is not None:
                va_pred, _ = self.forward(X_val, training=False)
                va_loss = self.loss(va_pred, y_val)
                self.history["val_loss"].append(va_loss)
                self.history["val_metric"].append(self._metric(X_val, y_val))

                if patience is not None:
                    if va_loss < best_val - 1e-6:
                        best_val, wait = va_loss, 0
                        best_state = ([w.copy() for w in self.W], [b.copy() for b in self.b])
                    else:
                        wait += 1
                        if wait >= patience:
                            if verbose_every:
                                print(f"  ⏹ Early stopping tại epoch {ep} (val_loss tốt nhất = {best_val:.4f})")
                            break

            if verbose_every and (ep % verbose_every == 0 or ep == 1):
                msg = f"  epoch {ep:4d} | train_loss={tr_loss:.4f} | train_metric={self.history['train_metric'][-1]:.4f}"
                if X_val is not None:
                    msg += f" | val_loss={self.history['val_loss'][-1]:.4f} | val_metric={self.history['val_metric'][-1]:.4f}"
                print(msg)

        if best_state is not None:
            self.W, self.b = best_state
        return self

    # ---------------------------- inference --------------------------------
    def predict_proba(self, X):
        out, _ = self.forward(X, training=False)
        return out

    def predict(self, X):
        out = self.predict_proba(X)
        if self.task == "binary":
            return (out >= 0.5).astype(int)
        if self.task == "multiclass":
            return out.argmax(1)
        return out

    # -------------------------- xuất ra JSON -------------------------------
    def n_params(self):
        return sum(w.size for w in self.W) + sum(b.size for b in self.b)

    def to_dict(self, decimals: int = 6):
        """Đóng gói trọng số về dict thuần Python để ghi ra model.json cho web."""
        return {
            "architecture": self.dims,
            "task": self.task,
            "activation": "relu",
            "n_params": int(self.n_params()),
            "layers": [
                {"W": np.round(w, decimals).tolist(), "b": np.round(bb, decimals).tolist()}
                for w, bb in zip(self.W, self.b)
            ],
        }


# ----------------------------------------------------------------------------
# 3. Tiện ích dùng chung
# ----------------------------------------------------------------------------


def one_hot(y, n_classes):
    out = np.zeros((len(y), n_classes))
    out[np.arange(len(y)), y] = 1.0
    return out


def forward_reference(model_dict, x):
    """Bản tham chiếu của thuật toán suy luận sẽ viết lại bằng JavaScript trên web.

    Dùng trong notebook để kiểm tra parity: NumPy <-> JSON <-> JavaScript.
    """
    a = np.asarray(x, dtype=float)
    layers = model_dict["layers"]
    for i, layer in enumerate(layers):
        z = a @ np.array(layer["W"]) + np.array(layer["b"])
        if i < len(layers) - 1:
            a = np.maximum(0.0, z)
        else:
            a = z
    if model_dict["task"] == "binary":
        return sigmoid(a)
    if model_dict["task"] == "multiclass":
        return softmax(a.reshape(1, -1))[0]
    return a

In [8]:
# ============================================================================
# KHỐI 8 — Chọn siêu tham số và lập bảng shape của 5 tầng
# Khác Bài toán 1 ở tầng cuối: head LINEAR (không kích hoạt) vì đây là bài hồi quy,
# đầu ra là một số thực trên thang z chứ không phải xác suất.
# Dropout để thấp (0.10) vì dữ liệu bảng ít nhiễu hơn và mạng dễ bị thiếu khớp.
# ============================================================================
HP = dict(lr=2e-3, l2=1e-4, dropout=0.10, batch_size=32, epochs=500, patience=60)
print("Siêu tham số (chọn trên tập VALIDATION theo R² của thang log):", HP)

mlp = DeepMLP(input_dim=Xtr.shape[1], hidden=(128, 64, 32, 16), output_dim=1,
              task="regression", lr=HP["lr"], l2=HP["l2"], dropout=HP["dropout"], seed=SEED)

rows = []
names = ["Layer 1 (Input→H1)", "Layer 2 (H1→H2)", "Layer 3 (H2→H3)",
         "Layer 4 (H3→H4)", "Layer 5 (H4→Output)"]
acts = ["ReLU", "ReLU", "ReLU", "ReLU", "Linear"]
for i, (nm, act) in enumerate(zip(names, acts)):
    fi, fo = mlp.dims[i], mlp.dims[i + 1]
    rows.append({"Tầng": nm, "W shape": f"({fi}, {fo})", "b shape": f"({fo},)",
                 "Output shape": f"(batch, {fo})", "Kích hoạt": act,
                 "Số tham số": fi * fo + fo})
shape_table = pd.DataFrame(rows)
shape_table.loc[len(shape_table)] = ["TỔNG", "", "", "", "", shape_table["Số tham số"].sum()]
shape_table

Siêu tham số (chọn trên tập VALIDATION theo R² của thang log): {'lr': 0.002, 'l2': 0.0001, 'dropout': 0.1, 'batch_size': 32, 'epochs': 500, 'patience': 60}


,Tầng,W shape,b shape,Output shape,Kích hoạt,Số tham số
0,Layer 1 (Input→H1),"(19, 128)","(128,)","(batch, 128)",ReLU,2560
1,Layer 2 (H1→H2),"(128, 64)","(64,)","(batch, 64)",ReLU,8256
2,Layer 3 (H2→H3),"(64, 32)","(32,)","(batch, 32)",ReLU,2080
3,Layer 4 (H3→H4),"(32, 16)","(16,)","(batch, 16)",ReLU,528
4,Layer 5 (H4→Output),"(16, 1)","(1,)","(batch, 1)",Linear,17
5,TỔNG,,,,,13441


In [9]:
# ============================================================================
# KHỐI 9 — Huấn luyện mạng NumPy trên thang log đã chuẩn hoá
# Hàm mất mát là MSE; early stopping theo val_loss với patience = 60.
# ============================================================================
t0 = time.perf_counter()
mlp.fit(Xtr, ztr, Xva, zva, epochs=HP["epochs"], batch_size=HP["batch_size"],
        verbose_every=50, patience=HP["patience"])
numpy_time = time.perf_counter() - t0
print(f"\n⏱ NumPy: {numpy_time:.2f}s | Tham số: {mlp.n_params():,}")

  epoch    1 | train_loss=0.2593 | train_metric=0.7407 | val_loss=0.2811 | val_metric=0.7073


  epoch   50 | train_loss=0.0840 | train_metric=0.9160 | val_loss=0.1223 | val_metric=0.8727


  ⏹ Early stopping tại epoch 78 (val_loss tốt nhất = 0.0846)

⏱ NumPy: 4.10s | Tham số: 13,441


## 4. Bản PyTorch tương đương

In [10]:
# ============================================================================
# KHỐI 10 — Bản PyTorch tương đương để đối chiếu
# Cùng kiến trúc, cùng Adam, nn.MSELoss; weight_decay được quy đổi
# l2 / batch_size cho khớp cách bản NumPy trung bình hoá số hạng phạt.
# ============================================================================
import torch
import torch.nn as nn

torch.manual_seed(SEED)
D = HP["dropout"]
torch_net = nn.Sequential(
    nn.Linear(Xtr.shape[1], 128), nn.ReLU(), nn.Dropout(D),
    nn.Linear(128, 64), nn.ReLU(), nn.Dropout(D),
    nn.Linear(64, 32), nn.ReLU(), nn.Dropout(D),
    nn.Linear(32, 16), nn.ReLU(), nn.Dropout(D),
    nn.Linear(16, 1),
)
# Quy đổi L2: bản NumPy chia gradient phạt cho batch_size, Adam của PyTorch thì không.
WEIGHT_DECAY = HP["l2"] / HP["batch_size"]
opt = torch.optim.Adam(torch_net.parameters(), lr=HP["lr"], weight_decay=WEIGHT_DECAY)
lossfn = nn.MSELoss()

Xtr_t = torch.tensor(Xtr, dtype=torch.float32); ztr_t = torch.tensor(ztr, dtype=torch.float32)
Xva_t = torch.tensor(Xva, dtype=torch.float32); zva_t = torch.tensor(zva, dtype=torch.float32)
Xte_t = torch.tensor(Xte, dtype=torch.float32)

torch_hist = {"train_loss": [], "val_loss": []}
best_state, best_val, wait = None, np.inf, 0
t0 = time.perf_counter()
n = len(Xtr_t)
for ep in range(1, HP["epochs"] + 1):
    torch_net.train()
    perm = torch.randperm(n)
    for s in range(0, n, HP["batch_size"]):
        idx = perm[s:s + HP["batch_size"]]
        opt.zero_grad()
        lossfn(torch_net(Xtr_t[idx]), ztr_t[idx]).backward()
        opt.step()
    torch_net.eval()
    with torch.no_grad():
        tl = lossfn(torch_net(Xtr_t), ztr_t).item()
        vl = lossfn(torch_net(Xva_t), zva_t).item()
    torch_hist["train_loss"].append(tl); torch_hist["val_loss"].append(vl)
    if vl < best_val - 1e-7:
        best_val, wait = vl, 0
        best_state = {k: v.clone() for k, v in torch_net.state_dict().items()}
    else:
        wait += 1
        if wait >= HP["patience"]:
            print(f"  ⏹ Early stopping tại epoch {ep}")
            break
    if ep % 50 == 0:
        print(f"  epoch {ep:4d} | train_mse={tl:.4f} | val_mse={vl:.4f}")
torch_net.load_state_dict(best_state)
torch_time = time.perf_counter() - t0
print(f"\n⏱ PyTorch: {torch_time:.2f}s")

  epoch   50 | train_mse=0.0390 | val_mse=0.0976


  ⏹ Early stopping tại epoch 67

⏱ PyTorch: 8.69s


## 5. Baseline Học máy truyền thống

Để so sánh sòng phẳng, **mọi mô hình đều học trên cùng thang log** và được quy
đổi ngược về giá thật bằng cùng một công thức.

In [11]:
# ============================================================================
# KHỐI 11 — Ba baseline hồi quy truyền thống
# Linear Regression, Decision Tree và Gradient Boosting học trên CÙNG thang log
# để việc so sánh với mạng nơ-ron là sòng phẳng (cùng đại lượng tối ưu hoá).
# ============================================================================
classical = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=8, random_state=SEED),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, max_depth=4,
                                                   learning_rate=0.08, random_state=SEED),
}
for name, m in classical.items():
    m.fit(Xtr, ztr.ravel())
    print(f"{name:20s} val R²(log) = {r2_score(zva.ravel(), m.predict(Xva)):.4f}")

Linear Regression    val R²(log) = 0.9072
Decision Tree        val R²(log) = 0.8450


Gradient Boosting    val R²(log) = 0.9112


## 6. Đánh giá trên tập kiểm thử

Ta báo cáo **hai bộ chỉ số**:

* **Thang log** — chính là đại lượng mô hình tối ưu hoá; đo sai số **tương đối**.
* **Thang giá thật (tỉ VNĐ)** — thứ người dùng cảm nhận; MAE/RMSE ở đây bị các
  bất động sản siêu đắt kéo lên rất mạnh.

In [ ]:
# ============================================================================
# KHỐI 12 — Đánh giá trên tập TEST theo hai thang đo
# Hàm evaluate() báo cáo song song:
#   - thang log: R2 và MAE, chính là đại lượng mà mô hình tối ưu hoá;
#   - thang giá: R2, MAE, RMSE, MAPE sau khi quy đổi ngược bằng to_price(),
#     tức con số mà người dùng cuối thực sự cảm nhận được (đơn vị tỉ VNĐ).
# ============================================================================
def evaluate(name, z_pred):
    z_pred = np.asarray(z_pred).reshape(-1, 1)
    p_pred = to_price(z_pred)
    p_true = y_test
    mape = float(np.mean(np.abs((p_pred - p_true) / p_true)) * 100)
    return {
        "Mô hình": name,
        "R² (log)": r2_score(zte.ravel(), z_pred.ravel()),
        "MAE (log)": mean_absolute_error(zte.ravel(), z_pred.ravel()),
        "R² (giá)": r2_score(p_true, p_pred),
        "MAE (tỉ)": mean_absolute_error(p_true, p_pred),
        "RMSE (tỉ)": float(np.sqrt(mean_squared_error(p_true, p_pred))),
        "MAPE (%)": mape,
    }

z_np = mlp.predict(Xte)
with torch.no_grad():
    z_torch = torch_net(Xte_t).numpy()

preds = {name: m.predict(Xte).reshape(-1, 1) for name, m in classical.items()}

preds["Deep MLP-5 (NumPy)"] = z_np
preds["Deep MLP-5 (PyTorch)"] = z_torch

res_df = pd.DataFrame([evaluate(k, v) for k, v in preds.items()]).set_index("Mô hình").round(4)
res_df.sort_values("R² (log)", ascending=False)

,R² (log),MAE (log),R² (giá),MAE (tỉ),RMSE (tỉ),MAPE (%)
Mô hình,,,,,,
Deep MLP-5 (PyTorch),0.9332,0.2250,0.8938,3.5937,5.9977,17.6865
Gradient Boosting,0.9299,0.2242,0.8836,3.5784,6.2785,17.2187
Deep MLP-5 (NumPy),0.9240,0.2364,0.8627,3.7382,6.8208,18.4613
Linear Regression,0.9148,0.2417,0.6621,4.2037,10.6988,18.9598
Decision Tree,0.8546,0.3158,0.6913,5.3740,10.2265,24.4083


## 7. Trực quan hoá

### 7.1. Đường cong huấn luyện hồi quy

In [13]:
# ============================================================================
# KHỐI 13 — Hình 2 — đường cong huấn luyện hồi quy
# (a) MSE train/val của bản NumPy vẽ trên trục y log để thấy rõ giai đoạn hội tụ.
# (b) R2 theo epoch.
# (c) MSE của bản PyTorch để đối chiếu.
# ============================================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

ep = range(1, len(mlp.history["train_loss"]) + 1)
axes[0].plot(ep, mlp.history["train_loss"], color="#2563eb", lw=2, label="Train MSE")
axes[0].plot(ep, mlp.history["val_loss"], color="#f97316", lw=2, label="Validation MSE")
best_ep = int(np.argmin(mlp.history["val_loss"])) + 1
axes[0].axvline(best_ep, ls="--", color="#16a34a", alpha=0.8)
axes[0].annotate(f"Val MSE thấp nhất\nepoch {best_ep}",
                 xy=(best_ep, min(mlp.history["val_loss"])),
                 xytext=(best_ep * 0.35 + 20, min(mlp.history["val_loss"]) + 0.25),
                 arrowprops=dict(arrowstyle="->", color="#16a34a"), fontsize=8, color="#16a34a")
axes[0].set_yscale("log")
axes[0].set_title("(a) MSE trên thang log — NumPy (trục y log)", fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE"); axes[0].legend()

axes[1].plot(ep, mlp.history["train_metric"], color="#2563eb", lw=2, label="Train R²")
axes[1].plot(ep, mlp.history["val_metric"], color="#f97316", lw=2, label="Validation R²")
axes[1].set_title("(b) Hệ số xác định R² theo epoch", fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("R²"); axes[1].legend()

ep2 = range(1, len(torch_hist["train_loss"]) + 1)
axes[2].plot(ep2, torch_hist["train_loss"], color="#7c3aed", lw=2, label="Train MSE (PyTorch)")
axes[2].plot(ep2, torch_hist["val_loss"], color="#dc2626", lw=2, label="Val MSE (PyTorch)")
axes[2].set_yscale("log")
axes[2].set_title("(c) Bản PyTorch — cùng kiến trúc", fontweight="bold")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("MSE"); axes[2].legend()

plt.tight_layout()
plt.savefig(FIG / "p2_fig2_curves.png", bbox_inches="tight")
plt.show()

C:\Users\admin\AppData\Local\Temp\ipykernel_3052\3753972388.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 7.2. Giá thực tế vs Giá dự đoán (thang log)

In [14]:
# ============================================================================
# KHỐI 14 — Hình 3 — giá thực tế so với giá dự đoán
# Scatter log–log của 3 mô hình; đường đỏ nét đứt là dự đoán hoàn hảo,
# hai đường xám là biên +-30%. Điểm càng bám sát đường chéo càng tốt.
# ============================================================================
fig, axes = plt.subplots(1, 3, figsize=(16.5, 5))
show = ["Linear Regression", "Gradient Boosting", "Deep MLP-5 (NumPy)"]
for ax, name in zip(axes, show):
    p_pred = to_price(preds[name]).ravel()
    p_true = y_test.ravel()
    ax.scatter(p_true, p_pred, s=16, alpha=0.5, color="#0ea5e9", edgecolor="none")
    lo, hi = min(p_true.min(), p_pred.min()), max(p_true.max(), p_pred.max())
    ax.plot([lo, hi], [lo, hi], "r--", lw=1.6, label="Dự đoán hoàn hảo")
    ax.plot([lo, hi], [lo * 1.3, hi * 1.3], color="#9ca3af", ls=":", lw=1, label="±30%")
    ax.plot([lo, hi], [lo / 1.3, hi / 1.3], color="#9ca3af", ls=":", lw=1)
    ax.set_xscale("log"); ax.set_yscale("log")
    r2 = res_df.loc[name, "R² (log)"]; mae = res_df.loc[name, "MAE (tỉ)"]
    ax.set_title(f"{name}\nR²(log)={r2:.3f} | MAE={mae:.2f} tỉ", fontweight="bold", fontsize=10)
    ax.set_xlabel("Giá thực tế (tỉ VNĐ, log)"); ax.set_ylabel("Giá dự đoán (tỉ VNĐ, log)")
    ax.legend(fontsize=7.5, loc="upper left")
plt.tight_layout()
plt.savefig(FIG / "p2_fig3_scatter.png", bbox_inches="tight")
plt.show()

C:\Users\admin\AppData\Local\Temp\ipykernel_3052\2137802178.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 7.3. Phân tích cấu trúc sai số — vì sao MAE/RMSE trên thang giá "bùng nổ"

In [15]:
# ============================================================================
# KHỐI 15 — Hình 4 — phân tích cấu trúc sai số
# Giải thích vì sao MAE/RMSE trên thang giá trông rất lớn:
# (a) sai số TUYỆT ĐỐI tăng theo mức giá (sai 10% của căn 100 tỉ là 10 tỉ);
# (b) sai số TƯƠNG ĐỐI gần như không đổi ở mọi mức giá — mô hình thực ra ổn định;
# (c) nhóm giá cao tuy chiếm ít mẫu nhưng đóng góp phần lớn tổng sai số.
# ============================================================================
p_pred = to_price(z_np).ravel()
p_true = y_test.ravel()
abs_err = np.abs(p_pred - p_true)
rel_err = abs_err / p_true * 100

fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.6))

# (a) sai số tuyệt đối theo mức giá
axes[0].scatter(p_true, abs_err, s=16, alpha=0.5, color="#ef4444", edgecolor="none")
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_title("(a) Sai số tuyệt đối tăng theo mức giá", fontweight="bold")
axes[0].set_xlabel("Giá thực tế (tỉ VNĐ, log)"); axes[0].set_ylabel("|Sai số| (tỉ VNĐ, log)")

# (b) sai số tương đối theo mức giá — gần như phẳng
axes[1].scatter(p_true, rel_err, s=16, alpha=0.5, color="#16a34a", edgecolor="none")
axes[1].axhline(np.median(rel_err), color="#111827", ls="--",
                label=f"Trung vị = {np.median(rel_err):.1f}%")
axes[1].set_xscale("log")
axes[1].set_ylim(0, min(300, rel_err.max()))
axes[1].set_title("(b) Sai số TƯƠNG ĐỐI gần như không đổi", fontweight="bold")
axes[1].set_xlabel("Giá thực tế (tỉ VNĐ, log)"); axes[1].set_ylabel("Sai số tương đối (%)")
axes[1].legend()

# (c) đóng góp vào tổng MAE theo nhóm giá
bins = [0, 5, 15, 40, 100, np.inf]
labels = ["<5", "5–15", "15–40", "40–100", ">100"]
grp = pd.cut(p_true, bins=bins, labels=labels)
contrib = pd.DataFrame({"err": abs_err, "grp": grp}).groupby("grp", observed=False)["err"]
share = (contrib.sum() / abs_err.sum() * 100)
cnt_share = (contrib.count() / len(abs_err) * 100)
x = np.arange(len(labels))
axes[2].bar(x - 0.2, cnt_share.values, 0.4, label="% số mẫu", color="#93c5fd")
axes[2].bar(x + 0.2, share.values, 0.4, label="% đóng góp vào tổng sai số", color="#dc2626")
for i, (a, b) in enumerate(zip(cnt_share.values, share.values)):
    axes[2].text(i - 0.2, a + 1, f"{a:.0f}%", ha="center", fontsize=8)
    axes[2].text(i + 0.2, b + 1, f"{b:.0f}%", ha="center", fontsize=8)
axes[2].set_xticks(x); axes[2].set_xticklabels(labels)
axes[2].set_xlabel("Nhóm giá (tỉ VNĐ)"); axes[2].set_ylabel("%")
axes[2].set_title("(c) Nhóm giá cao chiếm phần lớn tổng sai số", fontweight="bold")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIG / "p2_fig4_error.png", bbox_inches="tight")
plt.show()

print(f"MAE  trên thang giá : {abs_err.mean():.3f} tỉ")
print(f"Trung vị |sai số|   : {np.median(abs_err):.3f} tỉ")
print(f"MAPE                : {rel_err.mean():.2f}%")
print(f"Trung vị sai số %   : {np.median(rel_err):.2f}%")

MAE  trên thang giá : 3.738 tỉ
Trung vị |sai số|   : 2.222 tỉ
MAPE                : 18.46%
Trung vị sai số %   : 14.82%


C:\Users\admin\AppData\Local\Temp\ipykernel_3052\828109870.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 7.4. Biểu đồ đối sánh mô hình

In [16]:
# ============================================================================
# KHỐI 16 — Hình 5 — biểu đồ cột đối sánh các mô hình
# Ba bảng con cho R2 (log), MAE và MAPE, có ghi chú rõ chỉ số nào cao hơn là tốt hơn.
# ============================================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
order = list(preds.keys())
short = [o.replace(" (", "\n(") for o in order]
for ax, col, color in zip(axes, ["R² (log)", "MAE (tỉ)", "MAPE (%)"],
                          ["#2563eb", "#f97316", "#16a34a"]):
    vals = [res_df.loc[o, col] for o in order]
    bars = ax.bar(short, vals, color=color, width=0.6)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.3f}", ha="center",
                va="bottom", fontsize=8, fontweight="bold")
    ax.set_title(col + ("  (cao hơn = tốt hơn)" if col.startswith("R²") else "  (thấp hơn = tốt hơn)"),
                 fontweight="bold", fontsize=10)
    ax.tick_params(axis="x", labelsize=8)
plt.tight_layout()
plt.savefig(FIG / "p2_fig5_compare.png", bbox_inches="tight")
plt.show()

C:\Users\admin\AppData\Local\Temp\ipykernel_3052\3920757456.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Xuất mô hình và nhúng vào Web App

In [17]:
# ============================================================================
# KHỐI 17 — Xuất mô hình và tham số tiền xử lý ra model_deep.json
# Bundle lưu thêm danh sách tỉnh/loại hình (để web dựng lại đúng thứ tự one-hot)
# và target_transform (mean/std của log1p) để trang web quy đổi ngược ra giá thật.
# ============================================================================
deep_bundle = mlp.to_dict(decimals=6)
deep_bundle.update({
    "model_name": "Deep MLP 5 tầng (NumPy from scratch)",
    "feature_names": FEATURE_NAMES,
    "numeric_features": NUM,
    "provinces": provinces,
    "house_types": house_types,
    "scaler": {"mean_": scaler.mean_.tolist(), "scale_": scaler.scale_.tolist()},
    "target_transform": {"kind": "log1p_zscore", "mean": Y_MEAN, "std": Y_STD},
    "metrics": {
        "r2_log": float(res_df.loc["Deep MLP-5 (NumPy)", "R² (log)"]),
        "mae_log": float(res_df.loc["Deep MLP-5 (NumPy)", "MAE (log)"]),
        "r2_price": float(res_df.loc["Deep MLP-5 (NumPy)", "R² (giá)"]),
        "mae_price": float(res_df.loc["Deep MLP-5 (NumPy)", "MAE (tỉ)"]),
        "rmse_price": float(res_df.loc["Deep MLP-5 (NumPy)", "RMSE (tỉ)"]),
        "mape": float(res_df.loc["Deep MLP-5 (NumPy)", "MAPE (%)"]),
        "median_ape": float(np.median(rel_err)),
    },
    "baseline_metrics": {k: {c: float(res_df.loc[k, c]) for c in res_df.columns}
                         for k in classical},
    "training": {
        "epochs_run": len(mlp.history["train_loss"]),
        **{k: HP[k] for k in ["lr", "l2", "dropout", "batch_size"]},
        "numpy_seconds": round(numpy_time, 2),
        "pytorch_seconds": round(torch_time, 2),
    },
    "history": {
        "train_loss": [round(v, 5) for v in mlp.history["train_loss"]],
        "val_loss": [round(v, 5) for v in mlp.history["val_loss"]],
        "train_r2": [round(v, 5) for v in mlp.history["train_metric"]],
        "val_r2": [round(v, 5) for v in mlp.history["val_metric"]],
    },
})

out_path = ROOT / "model_deep.json"
out_path.write_text(json.dumps(deep_bundle, ensure_ascii=False, separators=(",", ":")),
                    encoding="utf-8")
print(f"✅ Đã ghi {out_path.name} — {out_path.stat().st_size/1024:.1f} KB")

✅ Đã ghi model_deep.json — 128.6 KB


### 8.1. Kiểm tra parity NumPy ↔ JSON

In [18]:
# ============================================================================
# KHỐI 18 — Kiểm tra parity NumPy và bundle JSON
# So sánh đầu ra trên thang z giữa mô hình gốc và bundle JSON trên toàn tập test;
# assert bảo đảm sai lệch nhỏ hơn 1e-5.
# ============================================================================
reloaded = json.loads(out_path.read_text(encoding="utf-8"))
max_err = 0.0
for i in range(len(Xte)):
    ref = float(np.ravel(forward_reference(reloaded, Xte[i]))[0])
    max_err = max(max_err, abs(ref - float(z_np[i, 0])))
print(f"Sai số lớn nhất (thang z) giữa mô hình gốc và bundle JSON: {max_err:.3e}")
assert max_err < 1e-5
print("✅ Parity PASSED")

Sai số lớn nhất (thang z) giữa mô hình gốc và bundle JSON: 3.433e-06
✅ Parity PASSED


### 8.2. Nhúng vào `index.html`

In [19]:
# ============================================================================
# KHỐI 19 — Nhúng bundle vào index.html
# Thay dòng const DEEP_MODEL = ...; bằng regex, giữ nguyên phần còn lại của file.
# ============================================================================
import re

html_path = ROOT / "index.html"
html = html_path.read_text(encoding="utf-8")
payload = json.dumps(deep_bundle, ensure_ascii=False, separators=(",", ":"))
if re.search(r"^const DEEP_MODEL = .*;$", html, flags=re.M):
    html = re.sub(r"^const DEEP_MODEL = .*;$",
                  lambda _: f"const DEEP_MODEL = {payload};", html, count=1, flags=re.M)
    html_path.write_text(html, encoding="utf-8")
    print(f"✅ Đã nhúng DEEP_MODEL vào index.html ({len(payload)/1024:.1f} KB)")
else:
    print("⚠ Không tìm thấy chốt `const DEEP_MODEL = ...;` — bỏ qua.")

✅ Đã nhúng DEEP_MODEL vào index.html (128.5 KB)


## 9. Kết luận Bài toán 2

In [20]:
# ============================================================================
# KHỐI 20 — Bảng xếp hạng cuối cùng theo R2 trên thang log
# ============================================================================
res_df.sort_values("R² (log)", ascending=False)

,R² (log),MAE (log),R² (giá),MAE (tỉ),RMSE (tỉ),MAPE (%)
Mô hình,,,,,,
Deep MLP-5 (PyTorch),0.9332,0.2250,0.8938,3.5937,5.9977,17.6865
Gradient Boosting,0.9299,0.2242,0.8836,3.5784,6.2785,17.2187
Deep MLP-5 (NumPy),0.9240,0.2364,0.8627,3.7382,6.8208,18.4613
Linear Regression,0.9148,0.2417,0.6621,4.2037,10.6988,18.9598
Decision Tree,0.8546,0.3158,0.6913,5.3740,10.2265,24.4083


In [21]:
# ============================================================================
# KHỐI 21 — In tóm tắt kết quả Bài toán 2
# Gom kiến trúc, số tham số, thời gian huấn luyện và các chỉ số ở cả hai thang đo.
# ============================================================================
print(f"""
TÓM TẮT BÀI TOÁN 2 — HOUSE PRICE PREDICTION
{'='*66}
Kiến trúc      : {Xtr.shape[1]} → 128 → 64 → 32 → 16 → 1 (Linear)
Tổng tham số   : {mlp.n_params():,}
Epoch đã chạy  : {len(mlp.history['train_loss'])} (early stopping, patience={HP['patience']})
Thời gian train: NumPy {numpy_time:.2f}s | PyTorch {torch_time:.2f}s
R² (thang log) : {res_df.loc['Deep MLP-5 (NumPy)', 'R² (log)']:.4f}
R² (thang giá) : {res_df.loc['Deep MLP-5 (NumPy)', 'R² (giá)']:.4f}
MAE            : {res_df.loc['Deep MLP-5 (NumPy)', 'MAE (tỉ)']:.3f} tỉ VNĐ
RMSE           : {res_df.loc['Deep MLP-5 (NumPy)', 'RMSE (tỉ)']:.3f} tỉ VNĐ
MAPE           : {res_df.loc['Deep MLP-5 (NumPy)', 'MAPE (%)']:.2f}%  (trung vị {np.median(rel_err):.2f}%)
{'='*66}
""")


TÓM TẮT BÀI TOÁN 2 — HOUSE PRICE PREDICTION
Kiến trúc      : 19 → 128 → 64 → 32 → 16 → 1 (Linear)
Tổng tham số   : 13,441
Epoch đã chạy  : 78 (early stopping, patience=60)
Thời gian train: NumPy 4.10s | PyTorch 8.69s
R² (thang log) : 0.9240
R² (thang giá) : 0.8627
MAE            : 3.738 tỉ VNĐ
RMSE           : 6.821 tỉ VNĐ
MAPE           : 18.46%  (trung vị 14.82%)

